In [ ]:
!git clone https://github.com/MarioAlessandroNapoli/neuro-llm.git
%cd neuro-llm
!pip install -q -r requirements.txt

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")

In [ ]:
from huggingface_hub import snapshot_download, whoami

hf_user = whoami()["name"]
snapshot_download(f"{hf_user}/tinystories-tokenized", repo_type="dataset", local_dir="data")

In [ ]:
# Smoke griglia 1a (D10): funzionale + velocita' in un colpo solo, gruppo bench.
# M2 abolito per i bracci a scan (misurato: 112 s/step su MPS). Niente --hub-repo:
# gli smoke non entrano nel registro. wrnn per ultimo e corto: il suo tok/s decide
# il calendario dei suoi seed (mixer escluso da compile, deviazione dichiarata).
GROUP = 'bench'
for arch in ['linoss', 'dlinoss', 'dlinoss-phi', 'hyb-oa', 'hyb-ao']:
    !python -m src.train --arch {arch} --tokens 5000000 --seed 1 --lr 1e-3 \
        --devices 2 --batch-size 16 --compile --group {GROUP} --max-time 00:00:30:00
!python -m src.train --arch wrnn --tokens 2000000 --seed 1 --lr 1e-3 \
    --devices 2 --batch-size 16 --compile --group {GROUP} --max-time 00:02:00:00


In [ ]:
# Alleggerisce il packaging di fine sessione (gotcha ml-dev).
!rm -rf checkpoints
